### reload modules

In [11]:
# run when changing code in .py file
%load_ext autoreload
%autoreload 2
# %aimport nlp.rag_1

# import os
# print(os.getcwd())

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# bert-1
- run pipeline

In [ ]:
from nlp import ClimateBERTAnalyzer, analyze_reports

stats = analyze_reports('../data/reports/Baosteel')
# stats = analyze_reports("../data/reports")

# bert-2
- run vizualisations

In [ ]:
from nlp import ClimateBERTVisualizer, visualize_results

visualize_results("../cache", "../out")

✅ Loaded: 15 reports, 1 companies, 2013-2020

EXPORTING CSV FILES
   ✓ company_year.csv (8 rows)
   ✓ company_totals.csv (1 companies)
   ✓ yearly_industry.csv (8 years)
   ✓ funnel_company_year.csv (8 rows)

   📁 All CSVs saved to: ../out/

GENERATING PLOTS
   ✓ slide_main.png
   ✓ slide_sentiment_trend.png
   ✓ talk_score_trend.png
   ✓ funnel_trend.png
   ✓ talk_score_per_company.png
   ✓ per_company_components.png
   ✓ per_company_sentiment.png
   ✓ sentiment_all_companies.png
   ✓ n0_funnel.png
   ✓ n0_quality_comparison.png
   ✓ n0_per_company.png
   ✓ n0_gap_analysis.png

   📁 All plots saved to: ../out/

   Generating word frequency plots...

   📊 ALL CHUNKS (top 30):
   environment(1232), iron(766), management(736), development(720), energy(628), technology(609), products(607), protection(559), production(525), industry(522), green(473), emission(453), system(428), reduction(421), base(410)

   🌱 OPPORTUNITY chunks (top 15):
   environment(809), development(574), technology(52

# rag 1

## Model Testing

Test which models work for extraction (format compliance + quality).

In [10]:
from nlp.rag_test import test_models, compare_extractions
from nlp import RAGConfig

# Base config (shared settings)
# llm_num_ctx: Ollama uses for actual context window, Groq uses for batch size calc only
base = dict(cache_dir="../cache", min_detector_score=0.7, max_chunks_per_group=7)

# Models to test - uncomment as needed
MODELS = [
    # Groq (cloud API) - context windows for batch calc
    RAGConfig(llm_provider="groq", model="llama-3.1-8b-instant", llm_num_ctx=128000, **base),
    # RAGConfig(llm_provider="groq", model="llama-3.3-70b-versatile", llm_num_ctx=128000, **base),
    # RAGConfig(llm_provider="groq", model="mixtral-8x7b-32768", llm_num_ctx=32768, **base),
    # RAGConfig(llm_provider="groq", model="gemma2-9b-it", llm_num_ctx=8192, **base),

    # Ollama (local) - context window actually used
    RAGConfig(llm_provider="ollama", model="qwen2.5:3b", llm_num_ctx=4096, **base),
    # RAGConfig(llm_provider="ollama", model="gemma3:4b", llm_num_ctx=4096, **base),
    # RAGConfig(llm_provider="ollama", model="gemma3:1b", llm_num_ctx=4096, **base),
    # RAGConfig(llm_provider="ollama", model="phi3:mini", llm_num_ctx=4096, **base),
]

results = test_models(MODELS, skip_extraction=False)
compare_extractions(results)

RAG MODEL TESTING (2 models) - 012/2021

--- groq/llama-3.1-8b-instant ---
    Format: PASS (0.3s)
    Output: [012_001]|||The high cost of green hydrogen remains a significant barrier. Production costs are 4-6x higher than grey hydrogen.
[012_002]|||Infrastruc...
✓ RAG Pipeline initialized (GPU: NVIDIA T1200 Laptop GPU (3.9GB))
✓ Filtered 1048 chunks below detector_score=0.7 (14545 remaining)
('001', '2022'): 503 chunks
('001', '2023'): 487 chunks
('001', '2021'): 484 chunks
('015', '2024'): 344 chunks
('001', '2019'): 338 chunks
('001', '2024'): 336 chunks
('001', '2020'): 332 chunks
('003', '2024'): 279 chunks
('015', '2025'): 277 chunks
('001', '2013'): 262 chunks
✓ Loaded 14545 chunks from 15 companies (../cache)
Loading Groq model: llama-3.1-8b-instant
    [DEBUG] Context preview (6056 chars):
    [012_005]
In addition, in order to improve the emissions, the extraction hood on the Coke Overhead Machine 1 (Investition 0.9 million €) was renewed and put into operation. Furthermore,

## Run Extraction

Configure and run the full extraction pipeline.

In [ ]:
from nlp import load_pipeline, RAGConfig
from nlp.rag_test import calc_batch_size

config = RAGConfig(
    llm_provider="groq",
    model="llama-3.1-8b-instant",
    llm_num_ctx=128000,  # Groq: for batch calc only, Ollama: actual context
    cache_dir="../cache",
    min_detector_score=0.7,
)

# Calculate optimal batch size based on context window
batch_info = calc_batch_size(config)
config.max_chunks_per_group = batch_info["max_chunks"]

pipeline = load_pipeline(config)

In [ ]:
# Extract all companies (saves to out/)
# pipeline.extract_all_companies()

# rag 2

In [ ]:
# from bertopic.representation import OpenAI,LlamaCPP
from nlp import TopicModelConfig, run_topic_modeling_pipeline

# Set True to ignore cached model/embeddings and retrain from scratch
FORCE_RETRAIN = True

config = TopicModelConfig(
    # Embedding model
    # embedding_model="sentence-transformers/all-mpnet-base-v2",
    embedding_model="BAAI/bge-small-en-v1.5",

    batch_size=64,

    # UMAP (dimensionality reduction)
    umap_n_neighbors=30,
    umap_n_components=15,
    umap_min_dist=0.05,
    umap_metric='cosine',
    umap_random_state=42,

    # HDBSCAN (clustering)
    hdbscan_min_cluster_size=10,  # Higher = fewer topics
    hdbscan_min_samples=2,        # Lower = less outliers
    hdbscan_metric='euclidean',
    hdbscan_cluster_selection_method='leaf',  # 'eom' (inclusive) or 'leaf' (tight, more cleanup required)

    # Vectorizer (c-TF-IDF)
    vectorizer_ngram_range=(1, 2),
    vectorizer_min_df=1,
    vectorizer_max_df=0.95,

    # Topic representation
    mmr_diversity=0.2, # 0 - pure relevance, redundant/simi word ... 1 - pure diverse. max diff word
    top_n_words=10,
    nr_topics=10,  # Set None for auto, or int to reduce post-hoc
    calculate_probabilities=True,

    # Outlier reduction (post-hoc)
    reduce_outliers=False,  # Reassign outliers to nearest topic
    reduce_outliers_strategy='embeddings',  # 'embeddings', 'c-tf-idf', or 'distributions'

    # Visualization UMAP (separate 2D projection)
    viz_umap_n_neighbors=10,
    viz_umap_n_components=2,
    viz_umap_min_dist=0.0,

    # LLM for topic labeling
    ollama_model="gemma3:4b",  # Fast + follows format. Avoid qwen3 (hidden thinking = slow)
    ollama_base_url="http://localhost:11434",
    llm_temperature=0.0,

    # Misc
    verbose=True,
    embeddings_cache_path=None,
)

results = run_topic_modeling_pipeline(
    data_folder="../out",
    output_folder="../out/topics5",
    config=config,
    force_retrain=FORCE_RETRAIN
)
# Access results
barriers_df = results['barriers']['df']
motivators_df = results['motivators']['df']

In [ ]:
motivators_df